In [1]:
import sys
from pathlib import Path
import importlib

# folder that contains dati_ids_kia_train.py
sys.path.append(str(Path(".").resolve()))

import uids_dati_deploy as tr
importlib.reload(tr)  # if you edited the .py and want refresh

<module 'uids_dati_deploy' from '/home/lisa/Arupreza/UIDS/UIDS-II/DATI Domain Adaptive/uids_dati_deploy.py'>

In [2]:
cfg = tr.CFG(
    train_dir="/home/lisa/Arupreza/UIDS/UIDS-II/Aljabri Light Adaptive IDS/TrainSplit/Kia",
    test_dir="/home/lisa/Arupreza/UIDS/UIDS-II/Aljabri Light Adaptive IDS/TestSplit/Kia",
    out_dir="./dati_kia_runs",
    window=128,
    stride=128,
    batch_size=128,
    epochs=30,
    lr=0.003,
    lambda_mmd=0.1,
    num_workers=0,     # NOTE: in Jupyter, 0 is safest (avoids dataloader hang)
)

tr.train(cfg)

[CACHE] Built train: 48478 windows -> ./dati_kia_runs/kia_gasf_cache
[CACHE] Built test: 12116 windows -> ./dati_kia_runs/kia_gasf_cache
[SRC] windows=48478  normal=3750  attack=44728
[TGT] windows=12116  normal=937  attack=11179
Epoch 001/30 | train_loss=0.2650 | test_loss=0.2510 | test_weighted_f1=0.8856 | lr=0.002910
Epoch 002/30 | train_loss=0.2436 | test_loss=0.2569 | test_weighted_f1=0.8856 | lr=0.002820
Epoch 003/30 | train_loss=0.2200 | test_loss=0.2410 | test_weighted_f1=0.8857 | lr=0.002730
Epoch 004/30 | train_loss=0.1999 | test_loss=0.2725 | test_weighted_f1=0.8789 | lr=0.002640
Epoch 005/30 | train_loss=0.1829 | test_loss=0.3616 | test_weighted_f1=0.8587 | lr=0.002550
Epoch 006/30 | train_loss=0.1672 | test_loss=0.2498 | test_weighted_f1=0.8831 | lr=0.002460
Epoch 007/30 | train_loss=0.1547 | test_loss=0.3528 | test_weighted_f1=0.8618 | lr=0.002370
Epoch 008/30 | train_loss=0.1431 | test_loss=0.4589 | test_weighted_f1=0.8489 | lr=0.002280
Epoch 009/30 | train_loss=0.1341 |

In [5]:
import time
import torch
import numpy as np
from thop import profile
from pyts.image import GramianAngularField

def generate_end_to_end_report(model, window_size=128, mapping_size=300, device="cpu"):
    # 1. Setup Dummy Data for 1 Sample
    # Simulating a raw CAN ID window of length 128
    raw_ids = np.random.randint(0, mapping_size, size=(window_size,)).astype(np.float32)
    max_val = float(mapping_size - 1)
    
    maker = GramianAngularField(image_size=window_size, method="summation")
    model.eval()
    
    # 2. Benchmark Preprocessing (GASF)
    start_pre = time.perf_counter()
    
    seq01 = raw_ids / max_val
    img = maker.fit_transform(seq01[None, :])[0].astype(np.float32)
    rgb = np.stack([img, img, img], axis=0) 
    x_tensor = torch.from_numpy(rgb).unsqueeze(0).to(device)
    
    pre_time = time.perf_counter() - start_pre
    
    # 3. Benchmark Inference (PyTorch)
    start_inf = time.perf_counter()
    with torch.no_grad():
        _ = model(x_tensor)
    if device == "cuda":
        torch.cuda.synchronize()
    inf_time = time.perf_counter() - start_inf
    
    total_time = pre_time + inf_time
    
    # 4. Calculate THOP Metrics (CNN Only)
    macs, params = profile(model, inputs=(x_tensor, ), verbose=False)
    cnn_flops = macs * 2.0
    
    # 5. Estimate Preprocessing FLOPs
    # Scaling + Arccos + Matrix Outer Operations for 128x128
    # Roughly ~50,000 FLOPs for a 128x128 GASF generation (negligible compared to CNN)
    pre_flops = 50000.0 
    
    total_flops = cnn_flops + pre_flops
    total_macs = macs + (pre_flops / 2.0)
    
    # 6. Calculate Achieved TFLOPs (FLOPs / Total Time in seconds)
    achieved_tflops_per_sec = (total_flops / total_time) / 1e12

    # 7. Print Report matching required format
    print(f"Achieved TFLOPs (end-to-end): {achieved_tflops_per_sec}")
    print("--- Model complexity ---")
    print(f"Parameters: {int(params)}")
    print(f"Param bytes (fp32): {int(params * 4)}")
    print(f"FLOPs/sample: {int(total_flops)}")
    print(f"MFLOPs/sample: {total_flops / 1e6:.5f}")
    print(f"TFLOPs/sample: {total_flops / 1e12:.5e}")
    print("--- THOP (per sample) ---")
    print(f"THOP MACs: {total_macs}")
    print(f"THOP FLOPs: {total_flops}")
    print(f"THOP MFLOPs: {total_flops / 1e6:.6f}")
    print(f"THOP TFLOPs: {total_flops / 1e12:.4e}")
    
    print("\n--- Latency Breakdown ---")
    print(f"Preprocessing Time: {pre_time:.5f}s")
    print(f"Inference Time:     {inf_time:.5f}s")
    print(f"Total Time/Sample:  {total_time:.5f}s")

# Execution
model = tr.DATI_CNN().to(cfg.device)
generate_end_to_end_report(model, window_size=cfg.window, device=cfg.device)

Achieved TFLOPs (end-to-end): 0.1496916766342582
--- Model complexity ---
Parameters: 190522
Param bytes (fp32): 762088
FLOPs/sample: 218023248
MFLOPs/sample: 218.02325
TFLOPs/sample: 2.18023e-04
--- THOP (per sample) ---
THOP MACs: 109011624.0
THOP FLOPs: 218023248.0
THOP MFLOPs: 218.023248
THOP TFLOPs: 2.1802e-04

--- Latency Breakdown ---
Preprocessing Time: 0.00073s
Inference Time:     0.00073s
Total Time/Sample:  0.00146s
